# CKD — ANOVA + Rank Fusion: Primary Analysis vs. Sensitivity Analysis

This notebook runs the **same pipeline twice** on the same underlying CKD data, so the two runs can be compared directly:

- **Primary Analysis**: the full 400-row dataset, with missing values handled by median (numerical) / mode (categorical) imputation, fit on training data only within each fold.
- **Sensitivity Analysis**: only the 158 complete-case rows (zero missing values anywhere), no imputation needed at all.

**Why this comparison matters, concretely**: dropping every row with a missing value would cost 60.5% of the dataset and visibly shifts the class balance (62.5%/37.5% CKD in the full data vs. 27.2%/72.8% in the complete cases) — a strong sign the missingness isn't random. If the Primary and Sensitivity results agree, that's real evidence the imputation approach isn't distorting the conclusions. If they disagree, that's equally real evidence — and tells you exactly where to be cautious.

**Methodology upgrade from the earlier `ckd_anova.ipynb`**: this version uses `doda.fusion.RankFusion` directly from the package — it's been implemented properly since then as genuine **Reciprocal Rank Fusion (RRF)**, `RRF(f) = 1/(k + math_rank(f)) + 1/(k + clinical_rank(f))`, the same algorithm from Cormack, Clarke & Buettcher (2009). No more locally-redefined fusion class needed.

**Also new**: feature-selection stability (25-run repeated stratified CV, Jaccard similarity) and CV predictive performance, both with **paired Wilcoxon signed-rank tests, Holm correction, and Cohen's d effect sizes** comparing ANOVA vs. DODA — following the same rigor established in the Heart Disease experiment, applied here to both Primary and Sensitivity data.

**Imbalance handling**: `class_weight="balanced"` / `scale_pos_weight` used throughout (unlike the Heart Disease notebook, which didn't need this) since CKD is meaningfully imbalanced (62.5/37.5) — consistent with how breast cancer's similar imbalance was handled earlier in this project.

In [1]:
# =============================================================================
# STEP 1: LOAD AND CLEAN RAW DATA (shared by both analyses)
# =============================================================================

import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/ckd.csv")
df = df.drop(columns=["id"])

# Known data-quality issues in this exact UCI file (see 01_ckd_eda.ipynb)
categorical_cols_raw = df.select_dtypes(include="object").columns
for col in categorical_cols_raw:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "?": np.nan})

for col in ["pcv", "wc", "rc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Encode categorical features and target
target_column = "target"

binary_maps = {
    "rbc":   {"normal": 0, "abnormal": 1},
    "pc":    {"normal": 0, "abnormal": 1},
    "pcc":   {"notpresent": 0, "present": 1},
    "ba":    {"notpresent": 0, "present": 1},
    "htn":   {"no": 0, "yes": 1},
    "dm":    {"no": 0, "yes": 1},
    "cad":   {"no": 0, "yes": 1},
    "appet": {"poor": 0, "good": 1},
    "pe":    {"no": 0, "yes": 1},
    "ane":   {"no": 0, "yes": 1},
}
for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

df[target_column] = df["classification"].map({"ckd": 1, "notckd": 0})
df = df.drop(columns=["classification"])

numerical_features = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
                       "sod", "pot", "hemo", "pcv", "wc", "rc"]
categorical_features = ["rbc", "pc", "pcc", "ba", "htn", "dm", "cad",
                         "appet", "pe", "ane"]

print("=" * 70)
print("CLEANED + ENCODED DATASET")
print("=" * 70)
print(f"Shape: {df.shape}")
display(df.head())

CLEANED + ENCODED DATASET
Shape: (400, 25)


C:\Users\johnm\AppData\Local\Temp\ipykernel_19772\1346706842.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_raw = df.select_dtypes(include="object").columns


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,target
0,48.0,80.0,1.020,1.0,0.0,NaN,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,NaN,0.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0,1


In [2]:
# =============================================================================
# STEP 2: DEFINE THE TWO DATASETS — PRIMARY (imputed) vs SENSITIVITY (complete-case)
# =============================================================================

# --- Primary: full dataset, missing values handled by imputation later ---
X_primary = df.drop(columns=[target_column])
y_primary = df[target_column]

# --- Sensitivity: complete cases only, no imputation needed ---
df_complete = df.dropna()
X_sensitivity = df_complete.drop(columns=[target_column])
y_sensitivity = df_complete[target_column]

print("=" * 70)
print("PRIMARY vs SENSITIVITY DATASET SIZES")
print("=" * 70)
print(f"Primary (full, to be imputed) : {X_primary.shape[0]} rows")
print(f"Sensitivity (complete-case)    : {X_sensitivity.shape[0]} rows "
      f"({X_sensitivity.shape[0]/X_primary.shape[0]*100:.1f}% of primary)")

print("\nClass balance comparison:")
print("Primary:")
display((y_primary.value_counts(normalize=True) * 100).round(2))
print("Sensitivity:")
display((y_sensitivity.value_counts(normalize=True) * 100).round(2))

PRIMARY vs SENSITIVITY DATASET SIZES
Primary (full, to be imputed) : 400 rows
Sensitivity (complete-case)    : 158 rows (39.5% of primary)

Class balance comparison:
Primary:


target
1    62.5
0    37.5
Name: proportion, dtype: float64

Sensitivity:


target
0    72.78
1    27.22
Name: proportion, dtype: float64

In [5]:
#%pip uninstall -y doda

In [6]:
#%pip install --no-cache-dir git+https://github.com/anandha-3679/DODA.git

In [7]:
# =============================================================================
# STEP 3: SHARED HELPER FUNCTIONS
# Defined once, called twice (Primary and Sensitivity) — avoids duplicating
# ~150 lines of loop logic twice with only the input data differing.
# =============================================================================

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

from doda import DODASelector
from doda.adapters.sklearn_adapter import SklearnAdapter
from doda.knowledge import JSONProvider
from doda.fusion import RankFusion

WEIGHTS_FILE = "../../../config/clinical_weights/ckd_clinical_weights.json"


def make_models(y_train):
    """Fresh model instances, class-imbalance-aware (CKD is ~63/37)."""
    return {
        "LR": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                      random_state=42, n_jobs=-1),
        "XGB": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
                              scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                              random_state=42, n_jobs=-1)
    }


def impute_if_needed(X_train, X_test, do_impute):
    """Median/mode imputation fit on TRAIN only — skipped entirely for the
    complete-case (Sensitivity) dataset, since it has no missing values."""
    if not do_impute:
        return X_train.copy(), X_test.copy()

    num_cols = [c for c in numerical_features if c in X_train.columns]
    cat_cols = [c for c in categorical_features if c in X_train.columns]

    num_imp = SimpleImputer(strategy="median")
    cat_imp = SimpleImputer(strategy="most_frequent")

    X_train_i, X_test_i = X_train.copy(), X_test.copy()
    X_train_i[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test_i[num_cols] = num_imp.transform(X_test[num_cols])
    X_train_i[cat_cols] = cat_imp.fit_transform(X_train[cat_cols])
    X_test_i[cat_cols] = cat_imp.transform(X_test[cat_cols])

    return X_train_i, X_test_i


def run_stability_analysis(X, y, k_values, do_impute, label):
    """25-run (5x5) repeated stratified CV. For each K, runs BOTH ANOVA and
    DODA, records selected feature sets, computes pairwise Jaccard similarity."""

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    all_jaccard = []

    for k in k_values:
        print(f"\n{'='*70}\n{label} STABILITY — TOP-{k}\n{'='*70}")

        for method in ["ANOVA", "DODA"]:
            selected_sets = []

            for train_idx, _ in cv.split(X, y):
                X_train = X.iloc[train_idx]
                y_train = y.iloc[train_idx]
                X_train_imp, _ = impute_if_needed(X_train, X_train, do_impute)

                if method == "ANOVA":
                    sel = SelectKBest(score_func=f_classif, k=k)
                    sel.fit(X_train_imp, y_train)
                    features = X_train_imp.columns[sel.get_support()].tolist()
                else:
                    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
                    provider = JSONProvider(WEIGHTS_FILE)
                    selector = DODASelector(operators=[operator], provider=provider,
                                             fusion=RankFusion(), top_k=k)
                    selector.fit(X_train_imp, y_train)
                    features = list(selector.get_selected_features())

                selected_sets.append(set(features))

            jaccard_scores = []
            for i in range(len(selected_sets)):
                for j in range(i + 1, len(selected_sets)):
                    inter = len(selected_sets[i] & selected_sets[j])
                    union = len(selected_sets[i] | selected_sets[j])
                    jaccard_scores.append(inter / union)

            for score in jaccard_scores:
                all_jaccard.append({"Top_K": k, "Method": method, "Jaccard": score})

            print(f"{method}: mean Jaccard = {np.mean(jaccard_scores):.4f} "
                  f"(std {np.std(jaccard_scores):.4f})")

    return pd.DataFrame(all_jaccard)


def run_cv_performance(X, y, k_values, do_impute, label):
    """25-run (5x5) repeated stratified CV predictive performance,
    ANOVA vs DODA, across 3 models and all Top-K values."""

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
    results = []

    for run_id, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        X_train_imp, X_test_imp = impute_if_needed(X_train, X_test, do_impute)

        for k in k_values:
            for method in ["ANOVA", "DODA"]:
                if method == "ANOVA":
                    sel = SelectKBest(score_func=f_classif, k=k)
                    X_train_sel = pd.DataFrame(sel.fit_transform(X_train_imp, y_train),
                                                columns=X_train_imp.columns[sel.get_support()])
                    X_test_sel = pd.DataFrame(sel.transform(X_test_imp),
                                               columns=X_train_sel.columns)
                else:
                    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
                    provider = JSONProvider(WEIGHTS_FILE)
                    selector = DODASelector(operators=[operator], provider=provider,
                                             fusion=RankFusion(), top_k=k)
                    X_train_sel = selector.fit_transform(X_train_imp, y_train)
                    X_test_sel = selector.transform(X_test_imp)

                models = make_models(y_train)
                for model_name, model in models.items():
                    if model_name == "LR":
                        scaler = StandardScaler()
                        Xtr = scaler.fit_transform(X_train_sel)
                        Xts = scaler.transform(X_test_sel)
                    else:
                        Xtr, Xts = X_train_sel, X_test_sel

                    model.fit(Xtr, y_train)
                    y_pred = model.predict(Xts)
                    y_prob = model.predict_proba(Xts)[:, 1]

                    results.append({
                        "Run": run_id, "Top_K": k, "Method": method, "Model": model_name,
                        "Accuracy": accuracy_score(y_test, y_pred),
                        "F1": f1_score(y_test, y_pred, zero_division=0),
                        "ROC_AUC": roc_auc_score(y_test, y_prob)
                    })

        if run_id % 5 == 0:
            print(f"{label}: completed {run_id}/25 CV runs")

    return pd.DataFrame(results)


def wilcoxon_holm_test(df, group_cols, value_col="Jaccard"):
    """Paired Wilcoxon signed-rank test (ANOVA vs DODA) + Cohen's d,
    with Holm correction across all comparisons in this dataframe."""
    rows = []
    for keys, group in df.groupby(group_cols):
        anova_vals = group[group.Method == "ANOVA"].sort_values(value_col)[value_col].values \
            if "Run" not in group.columns else \
            group[group.Method == "ANOVA"].sort_values("Run")[value_col].values
        doda_vals = group[group.Method == "DODA"].sort_values(value_col)[value_col].values \
            if "Run" not in group.columns else \
            group[group.Method == "DODA"].sort_values("Run")[value_col].values

        n = min(len(anova_vals), len(doda_vals))
        anova_vals, doda_vals = anova_vals[:n], doda_vals[:n]

        if np.allclose(anova_vals, doda_vals):
            stat, p = np.nan, 1.0
        else:
            try:
                stat, p = wilcoxon(anova_vals, doda_vals)
            except ValueError:
                stat, p = np.nan, 1.0

        diff = doda_vals - anova_vals
        pooled_std = np.std(np.concatenate([anova_vals, doda_vals]), ddof=1)
        cohens_d = diff.mean() / pooled_std if pooled_std > 0 else 0.0

        rows.append({
            **(dict(zip(group_cols, keys)) if isinstance(keys, tuple) else {group_cols[0]: keys}),
            "ANOVA_mean": anova_vals.mean(),
            "DODA_mean": doda_vals.mean(),
            "p_value": p,
            "cohens_d": cohens_d
        })

    result_df = pd.DataFrame(rows)
    if len(result_df) > 0:
        reject, p_adj, _, _ = multipletests(result_df["p_value"].fillna(1.0), method="holm")
        result_df["p_holm"] = p_adj
        result_df["significant"] = reject
    return result_df


print("Helper functions defined.")

ImportError: cannot import name 'RankFusion' from 'doda.fusion' (c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\fusion\__init__.py)

# PRIMARY ANALYSIS (Imputed, n=400)

## 1. Baseline (80/20 split, for direct comparison with earlier notebooks)

In [4]:
# =============================================================================
# PRIMARY: TRAIN-TEST SPLIT, IMPUTATION, SCALING
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_primary, y_primary, test_size=0.2, random_state=42, stratify=y_primary
)

X_train_imp, X_test_imp = impute_if_needed(X_train, X_test, do_impute=True)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("PRIMARY split:", X_train_scaled.shape, X_test_scaled.shape)
print("Missing after imputation:", X_train_scaled.isnull().sum().sum(),
      X_test_scaled.isnull().sum().sum())

PRIMARY split: (320, 24) (80, 24)
Missing after imputation: 0 0


In [5]:
# =============================================================================
# PRIMARY: ANOVA BASELINE TOP-K
# =============================================================================

k_values = [5, 10, 15, 20]

anova_selector = SelectKBest(score_func=f_classif, k="all")
anova_selector.fit(X_train_scaled, y_train)
anova_scores = pd.DataFrame({
    "Feature": X_train_scaled.columns, "ANOVA Score": anova_selector.scores_
}).sort_values("ANOVA Score", ascending=False).reset_index(drop=True)

anova_results = {}
for k in k_values:
    top_features = anova_scores.head(k)["Feature"].tolist()
    mask = X_train_scaled.columns.isin(top_features)
    anova_results[k] = {
        "features": top_features,
        "X_train": X_train_scaled.loc[:, mask],
        "X_test": X_test_scaled.loc[:, mask]
    }
    print(f"Top-{k}:", top_features)

Top-5: ['hemo', 'pcv', 'sg', 'rc', 'htn']
Top-10: ['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu']
Top-15: ['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu', 'pe', 'bgr', 'ane', 'sod', 'bp']
Top-20: ['hemo', 'pcv', 'sg', 'rc', 'htn', 'dm', 'al', 'pc', 'appet', 'bu', 'pe', 'bgr', 'ane', 'sod', 'bp', 'pcc', 'sc', 'rbc', 'su', 'cad']


In [6]:
# =============================================================================
# PRIMARY: BASELINE MODEL EVALUATION
# =============================================================================

primary_baseline_results = []
for k in k_values:
    Xtr, Xts = anova_results[k]["X_train"], anova_results[k]["X_test"]
    models = make_models(y_train)
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        primary_baseline_results.append({
            "Method": "ANOVA", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

primary_baseline_df = pd.DataFrame(primary_baseline_results)
display(primary_baseline_df)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,ANOVA,5,LR,0.9500,0.959184,0.986000
1,ANOVA,5,RF,0.9875,0.990099,0.998667
2,ANOVA,5,XGB,0.9875,0.990099,0.996000
3,ANOVA,10,LR,0.9750,0.979592,1.000000
4,ANOVA,10,RF,0.9875,0.989899,1.000000
5,ANOVA,10,XGB,0.9875,0.990099,1.000000
6,ANOVA,15,LR,0.9750,0.979592,1.000000
7,ANOVA,15,RF,1.0000,1.000000,1.000000
8,ANOVA,15,XGB,0.9875,0.989899,0.999333
9,ANOVA,20,LR,0.9750,0.979592,0.999333


In [7]:
# =============================================================================
# PRIMARY: RANK FUSION DODA
# =============================================================================

provider = JSONProvider(WEIGHTS_FILE)
fusion = RankFusion()

primary_doda_results_by_k = {}
for k in k_values:
    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
    selector = DODASelector(operators=[operator], provider=provider, fusion=fusion, top_k=k)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)
    features = selector.get_selected_features()
    primary_doda_results_by_k[k] = {"X_train": X_train_sel, "X_test": X_test_sel,
                                     "features": features, "selector": selector}
    print(f"Top-{k}:", features)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f6666bb47d0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.033768241197253), 'bp': np.float64(33.37626382493944), 'sg': np.float64(226.36654745916326), 'al': np.float64(121.28053233555288), 'su': np.float64(24.866490274599517), 'rbc': np.float64(26.176829268292586), 'pc': np.float64(54.83759124087606), 'pcc': np.float64(28.88664596273283), 'ba': np.float64(13.249999999999986), 'bgr': np.float64(47.41485748565791), 'bu': np.float64(53.326007627262655), 'sc': np.float64(27.84285915023033), 'sod': np.float64(37.26240983413097), 'pot': np.float64(0.11007057871478018), 'hemo': np.float64(379.74518553534807), 'pcv': np.float64(264.0615317634784), 'wc': np.float64(9.430401337813645), 'rc': np.float64(178.7101320341

In [8]:
# =============================================================================
# PRIMARY: DODA MODEL EVALUATION + COMBINED BASELINE COMPARISON
# =============================================================================

primary_doda_results = []
for k in k_values:
    Xtr, Xts = primary_doda_results_by_k[k]["X_train"], primary_doda_results_by_k[k]["X_test"]
    models = make_models(y_train)
    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        primary_doda_results.append({
            "Method": "ANOVA + Rank Fusion", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

primary_doda_df = pd.DataFrame(primary_doda_results)
primary_comparison_df = pd.concat([primary_baseline_df, primary_doda_df], ignore_index=True)
display(primary_comparison_df)

import os
os.makedirs("../../../results/ckd/primary", exist_ok=True)
primary_comparison_df.to_csv("../../../results/ckd/primary/anova_rankfusion_80_20_comparison.csv", index=False)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,ANOVA,5,LR,0.9500,0.959184,0.986000
1,ANOVA,5,RF,0.9875,0.990099,0.998667
2,ANOVA,5,XGB,0.9875,0.990099,0.996000
3,ANOVA,10,LR,0.9750,0.979592,1.000000
4,ANOVA,10,RF,0.9875,0.989899,1.000000
5,ANOVA,10,XGB,0.9875,0.990099,1.000000
6,ANOVA,15,LR,0.9750,0.979592,1.000000
7,ANOVA,15,RF,1.0000,1.000000,1.000000
8,ANOVA,15,XGB,0.9875,0.989899,0.999333
9,ANOVA,20,LR,0.9750,0.979592,0.999333


## 2. Feature-Selection Stability (25-run repeated CV)

In [9]:
# =============================================================================
# PRIMARY: STABILITY ANALYSIS
# =============================================================================

primary_jaccard_df = run_stability_analysis(
    X_primary, y_primary, k_values, do_impute=True, label="PRIMARY"
)

primary_stability_summary = primary_jaccard_df.groupby(["Top_K", "Method"])["Jaccard"].agg(
    ["mean", "std"]
).reset_index()
print("\n" + "=" * 70)
print("PRIMARY STABILITY SUMMARY")
print("=" * 70)
display(primary_stability_summary)


PRIMARY STABILITY — TOP-5


ANOVA: mean Jaccard = 0.8489 (std 0.1659)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f666f606960>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e339790>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(16.238582074320888), 'bp': np.float64(32.223206308274), 'sg': np.float64(241.42814152913004), 'al': np.float64(131.9976333802612), 'su': np.float64(28.80404802616907), 'rbc': np.float64(24.424698795180728), 'pc': np.float64(56.11764705882352), 'pcc': np.float64(25.295454545454547), 'ba': np.float64(11.077868852459018), 'bgr': np.float64(49.763225467006286), 'bu': np.float64(52.00828747927983), 'sc': np.float64(27.986061463592176), 'sod': np.float64(33.663215283452416), 'pot': np.float64(1.95615949837945), 'hemo': np.float64(395.13139919974367), 'pcv': np.float64(305.2140657119252), 'wc': np.float64(11.647345953010749), 'rc': np.float64(165.855157412474)

ANOVA: mean Jaccard = 0.8084 (std 0.1004)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f666f615f10>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171

DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(16.238582074320888), 'bp': np.float64(32.223206308274), 'sg': np.float64(241.42814152913004), 'al': np.float64(131.9976333802612), 'su': np.float64(28.80404802616907), 'rbc': np.float64(24.424698795180728), 'pc': np.float64(56.11764705882352), 'pcc': np.float64(25.295454545454547), 'ba': np.float64(11.077868852459018), 'bgr': np.float64(49.763225467006286), 'bu': np.float64(52.00828747927983), 'sc': np.float64(27.986061463592176), 'sod': np.float64(33.663215283452416), 'pot': np.float64(1.95615949837945), 'hemo': np.float64(395.13139919974367), 'pcv': np.float64(305.2140657119252), 'wc': np.float64(11.647345953010749), 'rc': np.float64(165.855157412474), 'htn': np.float64(178.875), 'dm': np.float64(142.83791208791212), 'cad': np.float64(19.412790697674414), 'appet': np.float64(57.416666666666664), 'pe': np

...

DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(22.050394248776605), 'bp': np.float64(27.797725249327783), 'sg': np.float64(244.66579240439916), 'al': np.float64(129.89057303825038), 'su': np.float64(31.65966753817356), 'rbc': np.float64(29.8125), 'pc': np.float64(56.11764705882352), 'pcc': np.float64(28.886645962732924), 'ba': np.float64(11.793956043956047), 'bgr': np.float64(57.185747172710954), 'bu': np.float64(50.02988432687374), 'sc': np.float64(28.567133583115695), 'sod': np.float64(38.95509804229048), 'pot': np.float64(1.1739180259812687), 'hemo': np.float64(331.9712165054786), 'pcv': np.float64(217.8926652957848), 'wc': np.float64(7.520962996127156), 'rc': np.float64(161.02745111444077), 'htn': np.float64(168.09939759036135), 'dm': np.float64(148.72752808988764), 'cad': np.float64(18.61127167630058), 'appet': np.float64(60.07330827067671), 'pe':

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e3afbf0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(17.41825923527365), 'bp': np.float64(32.97407698350354), 'sg': np.float64(236.5909344523254), 'al': np.float64(103.67542682000106), 'su': np.float64(28.549887892376677), 'rbc': np.float64(25.295454545454547), 'pc': np.float64(47.5332167832168), 'pcc': np.float64(21.874260355029584), 'ba': np.float64(10.369565217391305), 'bgr': np.float64(55.60616782524821), 'bu': np.float64(50.36496186468619), 'sc': np.float64(29.66584881079235), 'sod': np.float64(38.129241869384146), 'pot': np.float64(1.741555262815382), 'hemo': np.float64(376.0408411486195), 'pcv': np.float64(277.26843036683795), 'wc': np.float64(7.692369393305453), 'rc': np.float64(152.3825267631457)

ANOVA: mean Jaccard = 0.9600 (std 0.0470)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e34dbe0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e34c320>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(18.200503406752745), 'bp': np.float64(27.47020818377602), 'sg': np.float64(241.50039331167895), 'al': np.float64(121.23773701156738), 'su': np.float64(27.876623376623378), 'rbc': np.float64(27.069018404907975), 'pc': np.float64(48.707746478873254), 'pcc': np.float64(21.044117647058826), 'ba': np.float64(9.66891891891892), 'bgr': np.float64(53.31151146370995), 'bu': np.float64(47.83904006789586), 'sc': np.float64(26.28670598808899), 'sod': np.float64(38.49591722560613), 'pot': np.float64(0.7883780481406805), 'hemo': np.float64(354.0387122455604), 'pcv': np.float64(253.2636122326403), 'wc': np.float64(14.18235543364434), 'rc': np.float64(159.6223101234729

,Top_K,Method,mean,std
0,5,ANOVA,0.848889,0.166216
1,5,DODA,0.860000,0.164794
2,10,ANOVA,0.808384,0.100544
3,10,DODA,0.783784,0.127454
4,15,ANOVA,0.896814,0.063200
5,15,DODA,0.874534,0.075872
6,20,ANOVA,0.960000,0.047084
7,20,DODA,0.961270,0.046860


In [10]:
# =============================================================================
# PRIMARY: STABILITY SIGNIFICANCE TESTING (Wilcoxon + Holm + Cohen's d)
# =============================================================================

primary_stability_test = wilcoxon_holm_test(primary_jaccard_df, ["Top_K"], value_col="Jaccard")
print("=" * 70)
print("PRIMARY — ANOVA vs DODA STABILITY: SIGNIFICANCE TEST")
print("=" * 70)
display(primary_stability_test.round(4))

PRIMARY — ANOVA vs DODA STABILITY: SIGNIFICANCE TEST


,Top_K,ANOVA_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,0.8489,0.8600,0.0016,0.0672,0.0031,True
1,10,0.8084,0.7838,0.0000,-0.2133,0.0000,True
2,15,0.8968,0.8745,0.0000,-0.3153,0.0000,True
3,20,0.9600,0.9613,0.0455,0.0271,0.0455,True


## 3. Predictive Performance (25-run repeated CV)

In [11]:
# =============================================================================
# PRIMARY: CV PREDICTIVE PERFORMANCE
# =============================================================================

primary_cv_results = run_cv_performance(
    X_primary, y_primary, k_values, do_impute=True, label="PRIMARY"
)

primary_cv_summary = primary_cv_results.groupby(["Top_K", "Method", "Model"]).agg(
    {"Accuracy": ["mean", "std"], "F1": ["mean", "std"], "ROC_AUC": ["mean", "std"]}
).reset_index()
primary_cv_summary.columns = ["Top_K", "Method", "Model", "Acc_Mean", "Acc_STD",
                               "F1_Mean", "F1_STD", "AUC_Mean", "AUC_STD"]

os.makedirs("../../../results/ckd/primary", exist_ok=True)
primary_cv_results.to_csv("../../../results/ckd/primary/cv_performance_runs.csv", index=False)
primary_cv_summary.to_csv("../../../results/ckd/primary/cv_performance_summary.csv", index=False)

print("=" * 70)
print("PRIMARY CV PERFORMANCE SUMMARY")
print("=" * 70)
display(primary_cv_summary.round(4))

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e306570>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171), 'rc': np.float64(130.5421947828456), 'h

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e326930>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171), 'rc': np.float64(130.5421947828456), 'h

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e22c0e0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171), 'rc': np.float64(130.5421947828456), 'h

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e307fe0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(13.680931940287804), 'bp': np.float64(37.305673867208945), 'sg': np.float64(223.18444977221588), 'al': np.float64(123.19658076503616), 'su': np.float64(30.789409729824396), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(44.10616438356164), 'pcc': np.float64(20.22368421052632), 'ba': np.float64(7.611702127659573), 'bgr': np.float64(52.02822635558733), 'bu': np.float64(46.60060505313), 'sc': np.float64(42.10784534352323), 'sod': np.float64(75.86032790605188), 'pot': np.float64(1.16494223458721), 'hemo': np.float64(334.76933360080085), 'pcv': np.float64(256.140163846139), 'wc': np.float64(9.409275235718171), 'rc': np.float64(130.5421947828456), 'h

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f66675cd760>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(22.94798193690508), 'bp': np.float64(26.93377237833337), 'sg': np.float64(242.81183150298074), 'al': np.float64(118.62494876476508), 'su': np.float64(31.525657216302243), 'rbc': np.float64(28.886645962732924), 'pc': np.float64(53.576086956521735), 'pcc': np.float64(25.295454545454547), 'ba': np.float64(11.077868852459018), 'bgr': np.float64(56.91502759459745), 'bu': np.float64(49.47656866372859), 'sc': np.float64(30.634056849704766), 'sod': np.float64(37.55031341469965), 'pot': np.float64(1.9535675363810006), 'hemo': np.float64(355.5538465884311), 'pcv': np.float64(257.96415801012654), 'wc': np.float64(8.78354650671208), 'rc': np.float64(148.37510630790

...

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e35c6e0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(17.41825923527365), 'bp': np.float64(32.97407698350354), 'sg': np.float64(236.5909344523254), 'al': np.float64(103.67542682000106), 'su': np.float64(28.549887892376677), 'rbc': np.float64(25.295454545454547), 'pc': np.float64(47.5332167832168), 'pcc': np.float64(21.874260355029584), 'ba': np.float64(10.369565217391305), 'bgr': np.float64(55.60616782524821), 'bu': np.float64(50.36496186468619), 'sc': np.float64(29.66584881079235), 'sod': np.float64(38.129241869384146), 'pot': np.float64(1.741555262815382), 'hemo': np.float64(376.0408411486195), 'pcv': np.float64(277.26843036683795), 'wc': np.float64(7.692369393305453), 'rc': np.float64(152.3825267631457)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e34d580>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(17.41825923527365), 'bp': np.float64(32.97407698350354), 'sg': np.float64(236.5909344523254), 'al': np.float64(103.67542682000106), 'su': np.float64(28.549887892376677), 'rbc': np.float64(25.295454545454547), 'pc': np.float64(47.5332167832168), 'pcc': np.float64(21.874260355029584), 'ba': np.float64(10.369565217391305), 'bgr': np.float64(55.60616782524821), 'bu': np.float64(50.36496186468619), 'sc': np.float64(29.66584881079235), 'sod': np.float64(38.129241869384146), 'pot': np.float64(1.741555262815382), 'hemo': np.float64(376.0408411486195), 'pcv': np.float64(277.26843036683795), 'wc': np.float64(7.692369393305453), 'rc': np.float64(152.3825267631457)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e3ad910>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(17.41825923527365), 'bp': np.float64(32.97407698350354), 'sg': np.float64(236.5909344523254), 'al': np.float64(103.67542682000106), 'su': np.float64(28.549887892376677), 'rbc': np.float64(25.295454545454547), 'pc': np.float64(47.5332167832168), 'pcc': np.float64(21.874260355029584), 'ba': np.float64(10.369565217391305), 'bgr': np.float64(55.60616782524821), 'bu': np.float64(50.36496186468619), 'sc': np.float64(29.66584881079235), 'sod': np.float64(38.129241869384146), 'pot': np.float64(1.741555262815382), 'hemo': np.float64(376.0408411486195), 'pcv': np.float64(277.26843036683795), 'wc': np.float64(7.692369393305453), 'rc': np.float64(152.3825267631457)

PRIMARY: completed 25/25 CV runs
PRIMARY CV PERFORMANCE SUMMARY


,Top_K,Method,Model,Acc_Mean,Acc_STD,F1_Mean,F1_STD,AUC_Mean,AUC_STD
0,5,ANOVA,LR,0.9750,0.0153,0.9797,0.0126,0.9955,0.0039
1,5,ANOVA,RF,0.9855,0.0112,0.9884,0.0091,0.9990,0.0011
2,5,ANOVA,XGB,0.9775,0.0149,0.9820,0.0119,0.9986,0.0014
3,5,DODA,LR,0.9585,0.0283,0.9651,0.0248,0.9948,0.0057
4,5,DODA,RF,0.9705,0.0219,0.9760,0.0187,0.9956,0.0062
5,5,DODA,XGB,0.9695,0.0198,0.9754,0.0162,0.9947,0.0065
6,10,ANOVA,LR,0.9890,0.0083,0.9911,0.0068,0.9999,0.0002
7,10,ANOVA,RF,0.9910,0.0092,0.9928,0.0073,0.9996,0.0007
8,10,ANOVA,XGB,0.9785,0.0147,0.9828,0.0117,0.9983,0.0017
9,10,DODA,LR,0.9860,0.0110,0.9886,0.0091,0.9999,0.0002


In [12]:
# =============================================================================
# PRIMARY: PERFORMANCE SIGNIFICANCE TESTING
# =============================================================================

primary_perf_test = wilcoxon_holm_test(
    primary_cv_results, ["Top_K", "Model"], value_col="ROC_AUC"
)
print("=" * 70)
print("PRIMARY — ANOVA vs DODA ROC-AUC: SIGNIFICANCE TEST")
print("=" * 70)
display(primary_perf_test.round(4))

PRIMARY — ANOVA vs DODA ROC-AUC: SIGNIFICANCE TEST


,Top_K,Model,ANOVA_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,LR,0.9955,0.9948,1.0000,-0.1507,1.0000,False
1,5,RF,0.9990,0.9956,0.0105,-0.7136,0.1156,False
2,5,XGB,0.9986,0.9947,0.0043,-0.7677,0.0512,False
3,10,LR,0.9999,0.9999,0.1573,-0.2437,1.0000,False
4,10,RF,0.9996,0.9994,0.5106,-0.2338,1.0000,False
5,10,XGB,0.9983,0.9980,0.4499,-0.1674,1.0000,False
6,15,LR,0.9999,0.9999,1.0000,0.0000,1.0000,False
7,15,RF,0.9997,0.9999,0.2041,0.3535,1.0000,False
8,15,XGB,0.9980,0.9989,0.0120,0.4473,0.1196,False
9,20,LR,1.0000,1.0000,1.0000,0.0000,1.0000,False


# SENSITIVITY ANALYSIS (Complete-Case, n=158)

Identical pipeline, applied to the 158 rows with zero missing values — no imputation step at all. If the Primary and Sensitivity conclusions agree, the imputation choice wasn't driving the results.

## 1. Baseline (80/20 split)

In [13]:
# =============================================================================
# SENSITIVITY: TRAIN-TEST SPLIT + SCALING (no imputation needed)
# =============================================================================

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_sensitivity, y_sensitivity, test_size=0.2, random_state=42, stratify=y_sensitivity
)

scaler_s = StandardScaler()
Xs_train_scaled = pd.DataFrame(scaler_s.fit_transform(Xs_train), columns=Xs_train.columns)
Xs_test_scaled = pd.DataFrame(scaler_s.transform(Xs_test), columns=Xs_test.columns)
ys_train = ys_train.reset_index(drop=True)
ys_test = ys_test.reset_index(drop=True)

print("SENSITIVITY split:", Xs_train_scaled.shape, Xs_test_scaled.shape)

SENSITIVITY split: (126, 24) (32, 24)


In [14]:
# =============================================================================
# SENSITIVITY: ANOVA BASELINE TOP-K
# =============================================================================

anova_selector_s = SelectKBest(score_func=f_classif, k="all")
anova_selector_s.fit(Xs_train_scaled, ys_train)
anova_scores_s = pd.DataFrame({
    "Feature": Xs_train_scaled.columns, "ANOVA Score": anova_selector_s.scores_
}).sort_values("ANOVA Score", ascending=False).reset_index(drop=True)

anova_results_s = {}
for k in k_values:
    top_features = anova_scores_s.head(k)["Feature"].tolist()
    mask = Xs_train_scaled.columns.isin(top_features)
    anova_results_s[k] = {
        "features": top_features,
        "X_train": Xs_train_scaled.loc[:, mask],
        "X_test": Xs_test_scaled.loc[:, mask]
    }
    print(f"Top-{k}:", top_features)

Top-5: ['al', 'htn', 'pcv', 'hemo', 'sg']
Top-10: ['al', 'htn', 'pcv', 'hemo', 'sg', 'pc', 'rc', 'dm', 'sc', 'bu']
Top-15: ['al', 'htn', 'pcv', 'hemo', 'sg', 'pc', 'rc', 'dm', 'sc', 'bu', 'appet', 'sod', 'rbc', 'pe', 'bgr']
Top-20: ['al', 'htn', 'pcv', 'hemo', 'sg', 'pc', 'rc', 'dm', 'sc', 'bu', 'appet', 'sod', 'rbc', 'pe', 'bgr', 'pcc', 'ane', 'su', 'ba', 'cad']


In [15]:
# =============================================================================
# SENSITIVITY: BASELINE MODEL EVALUATION
# =============================================================================

sensitivity_baseline_results = []
for k in k_values:
    Xtr, Xts = anova_results_s[k]["X_train"], anova_results_s[k]["X_test"]
    models = make_models(ys_train)
    for model_name, model in models.items():
        model.fit(Xtr, ys_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        sensitivity_baseline_results.append({
            "Method": "ANOVA", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(ys_test, y_pred),
            "F1 Score": f1_score(ys_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(ys_test, y_prob)
        })

sensitivity_baseline_df = pd.DataFrame(sensitivity_baseline_results)
display(sensitivity_baseline_df)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,ANOVA,5,LR,1.00000,1.000000,1.0
1,ANOVA,5,RF,1.00000,1.000000,1.0
2,ANOVA,5,XGB,0.96875,0.941176,1.0
3,ANOVA,10,LR,1.00000,1.000000,1.0
4,ANOVA,10,RF,1.00000,1.000000,1.0
5,ANOVA,10,XGB,0.96875,0.941176,1.0
6,ANOVA,15,LR,1.00000,1.000000,1.0
7,ANOVA,15,RF,1.00000,1.000000,1.0
8,ANOVA,15,XGB,0.96875,0.941176,1.0
9,ANOVA,20,LR,1.00000,1.000000,1.0


In [16]:
# =============================================================================
# SENSITIVITY: RANK FUSION DODA + EVALUATION
# =============================================================================

sensitivity_doda_results_by_k = {}
for k in k_values:
    operator = SklearnAdapter(SelectKBest(score_func=f_classif, k="all"))
    selector = DODASelector(operators=[operator], provider=provider, fusion=RankFusion(), top_k=k)
    X_train_sel = selector.fit_transform(Xs_train_scaled, ys_train)
    X_test_sel = selector.transform(Xs_test_scaled)
    features = selector.get_selected_features()
    sensitivity_doda_results_by_k[k] = {"X_train": X_train_sel, "X_test": X_test_sel, "features": features}
    print(f"Top-{k}:", features)

sensitivity_doda_results = []
for k in k_values:
    Xtr, Xts = sensitivity_doda_results_by_k[k]["X_train"], sensitivity_doda_results_by_k[k]["X_test"]
    models = make_models(ys_train)
    for model_name, model in models.items():
        model.fit(Xtr, ys_train)
        y_pred, y_prob = model.predict(Xts), model.predict_proba(Xts)[:, 1]
        sensitivity_doda_results.append({
            "Method": "ANOVA + Rank Fusion", "Top-K": k, "Model": model_name,
            "Accuracy": accuracy_score(ys_test, y_pred),
            "F1 Score": f1_score(ys_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(ys_test, y_prob)
        })

sensitivity_doda_df = pd.DataFrame(sensitivity_doda_results)
sensitivity_comparison_df = pd.concat([sensitivity_baseline_df, sensitivity_doda_df], ignore_index=True)
display(sensitivity_comparison_df)

os.makedirs("../../../results/ckd/sensitivity", exist_ok=True)
sensitivity_comparison_df.to_csv("../../../results/ckd/sensitivity/anova_rankfusion_80_20_comparison.csv", index=False)

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f6666bb47d0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(14.297231773208832), 'bp': np.float64(16.81422987955652), 'sg': np.float64(203.6901803675013), 'al': np.float64(891.1386076740316), 'su': np.float64(45.53665762310162), 'rbc': np.float64(71.47869674185459), 'pc': np.float64(189.3102453102453), 'pcc': np.float64(56.04837490551773), 'ba': np.float64(37.724867724867714), 'bgr': np.float64(59.3720943634911), 'bu': np.float64(98.10100401458176), 'sc': np.float64(123.05571986557874), 'sod': np.float64(78.94197209172668), 'pot': np.float64(3.3654097354611974), 'hemo': np.float64(247.33202322732737), 'pcv': np.float64(247.82788174590155), 'wc': np.float64(18.79739354214816), 'rc': np.float64(135.84030002587076)

,Method,Top-K,Model,Accuracy,F1 Score,ROC-AUC
0,ANOVA,5,LR,1.00000,1.000000,1.0
1,ANOVA,5,RF,1.00000,1.000000,1.0
2,ANOVA,5,XGB,0.96875,0.941176,1.0
3,ANOVA,10,LR,1.00000,1.000000,1.0
4,ANOVA,10,RF,1.00000,1.000000,1.0
5,ANOVA,10,XGB,0.96875,0.941176,1.0
6,ANOVA,15,LR,1.00000,1.000000,1.0
7,ANOVA,15,RF,1.00000,1.000000,1.0
8,ANOVA,15,XGB,0.96875,0.941176,1.0
9,ANOVA,20,LR,1.00000,1.000000,1.0


## 2. Feature-Selection Stability (25-run repeated CV)

In [17]:
# =============================================================================
# SENSITIVITY: STABILITY ANALYSIS
# =============================================================================

sensitivity_jaccard_df = run_stability_analysis(
    X_sensitivity, y_sensitivity, k_values, do_impute=False, label="SENSITIVITY"
)

sensitivity_stability_summary = sensitivity_jaccard_df.groupby(["Top_K", "Method"])["Jaccard"].agg(
    ["mean", "std"]
).reset_index()
print("\n" + "=" * 70)
print("SENSITIVITY STABILITY SUMMARY")
print("=" * 70)
display(sensitivity_stability_summary)


SENSITIVITY STABILITY — TOP-5
ANOVA: mean Jaccard = 0.7833 (std 0.1590)
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e3046e0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(10.655838595793952), 'bp': np.float64(20.010236819882145), 'sg': np.float64(215.49980646808285), 'al': np.float64(896.8553459119493), 'su': np.float64(47.11007871983482), 'rbc': np.float64(63.37777777777779), 'pc': np.float64(165.98941798941797), 'pcc': np.float64(43.3015873015873), 'ba': np.float64(43.3015873015873), 'bgr': np.float64(56.268438004244295), 'bu': np.float64(106.02689573659777), 'sc': np.float64(126.1858727117594), 'sod': np.float64(81.79112984436362), 'pot': np.float64(4.282811103195126), 'hemo': np.float64(265.19343917266224), 'pcv': np.float64(246.16690559453045),

DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(14.057814049605197), 'bp': np.float64(17.28364583104029), 'sg': np.float64(194.49999587650058), 'al': np.float64(1190.9448818897642), 'su': np.float64(50.30621172353455), 'rbc': np.float64(60.36745406824148), 'pc': np.float64(261.5923009623796), 'pcc': np.float64(41.50262467191601), 'ba': np.float64(41.50262467191601), 'bgr': np.float64(77.14399863493536), 'bu': np.float64(113.80827083608544), 'sc': np.float64(145.9742818658552), 'sod': np.float64(78.80655655016432), 'pot': np.float64(4.058523803749385), 'hemo': np.float64(276.0173458789997), 'pcv': np.float64(265.12840140623865), 'wc': np.float64(17.771842750934134), 'rc': np.float64(133.88364686021492), 'htn': np.float64(305.61023622047253), 'dm': np.float64(153.24046032707452), 'cad': np.float64(31.344639612356154), 'appet': np.float64(76.25362619146303

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e310f20>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(18.070151226541917), 'bp': np.float64(13.546306445798045), 'sg': np.float64(232.7127342629038), 'al': np.float64(697.0530578017365), 'su': np.float64(47.296849087893875), 'rbc': np.float64(71.47869674185463), 'pc': np.float64(189.31024531024528), 'pcc': np.float64(37.724867724867735), 'ba': np.float64(23.473251028806583), 'bgr': np.float64(76.54785177917312), 'bu': np.float64(126.29725034637727), 'sc': np.float64(148.56916274330328), 'sod': np.float64(96.27634965207797), 'pot': np.float64(3.100218839884039), 'hemo': np.float64(283.8873641970172), 'pcv': np.float64(262.01825976915393), 'wc': np.float64(16.94847000587958), 'rc': np.float64(127.58075285622

,Top_K,Method,mean,std
0,5,ANOVA,0.783333,0.159256
1,5,DODA,0.763651,0.163215
2,10,ANOVA,0.972121,0.065620
3,10,DODA,0.910808,0.095570
4,15,ANOVA,0.927966,0.073419
5,15,DODA,0.922083,0.060670
6,20,ANOVA,0.963810,0.046305
7,20,DODA,0.984791,0.035692


In [18]:
sensitivity_stability_test = wilcoxon_holm_test(sensitivity_jaccard_df, ["Top_K"], value_col="Jaccard")
print("=" * 70)
print("SENSITIVITY — ANOVA vs DODA STABILITY: SIGNIFICANCE TEST")
print("=" * 70)
display(sensitivity_stability_test.round(4))

SENSITIVITY — ANOVA vs DODA STABILITY: SIGNIFICANCE TEST


,Top_K,ANOVA_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,0.7833,0.7637,0.0001,-0.1219,0.0001,True
1,10,0.9721,0.9108,0.0000,-0.7010,0.0000,True
2,15,0.9280,0.9221,0.0000,-0.0873,0.0000,True
3,20,0.9638,0.9848,0.0000,0.4923,0.0000,True


## 3. Predictive Performance (25-run repeated CV)

In [19]:
# =============================================================================
# SENSITIVITY: CV PREDICTIVE PERFORMANCE
# =============================================================================

sensitivity_cv_results = run_cv_performance(
    X_sensitivity, y_sensitivity, k_values, do_impute=False, label="SENSITIVITY"
)

sensitivity_cv_summary = sensitivity_cv_results.groupby(["Top_K", "Method", "Model"]).agg(
    {"Accuracy": ["mean", "std"], "F1": ["mean", "std"], "ROC_AUC": ["mean", "std"]}
).reset_index()
sensitivity_cv_summary.columns = ["Top_K", "Method", "Model", "Acc_Mean", "Acc_STD",
                                   "F1_Mean", "F1_STD", "AUC_Mean", "AUC_STD"]

sensitivity_cv_results.to_csv("../../../results/ckd/sensitivity/cv_performance_runs.csv", index=False)
sensitivity_cv_summary.to_csv("../../../results/ckd/sensitivity/cv_performance_summary.csv", index=False)

print("=" * 70)
print("SENSITIVITY CV PERFORMANCE SUMMARY")
print("=" * 70)
display(sensitivity_cv_summary.round(4))

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e310380>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(10.655838595793952), 'bp': np.float64(20.010236819882145), 'sg': np.float64(215.49980646808285), 'al': np.float64(896.8553459119493), 'su': np.float64(47.11007871983482), 'rbc': np.float64(63.37777777777779), 'pc': np.float64(165.98941798941797), 'pcc': np.float64(43.3015873015873), 'ba': np.float64(43.3015873015873), 'bgr': np.float64(56.268438004244295), 'bu': np.float64(106.02689573659777), 'sc': np.float64(126.1858727117594), 'sod': np.float64(81.79112984436362), 'pot': np.float64(4.282811103195126), 'hemo': np.float64(265.19343917266224), 'pcv': np.float64(246.16690559453045), 'wc': np.float64(19.442229376915883), 'rc': np.float64(117.6242605207385

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e3732c0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(10.655838595793952), 'bp': np.float64(20.010236819882145), 'sg': np.float64(215.49980646808285), 'al': np.float64(896.8553459119493), 'su': np.float64(47.11007871983482), 'rbc': np.float64(63.37777777777779), 'pc': np.float64(165.98941798941797), 'pcc': np.float64(43.3015873015873), 'ba': np.float64(43.3015873015873), 'bgr': np.float64(56.268438004244295), 'bu': np.float64(106.02689573659777), 'sc': np.float64(126.1858727117594), 'sod': np.float64(81.79112984436362), 'pot': np.float64(4.282811103195126), 'hemo': np.float64(265.19343917266224), 'pcv': np.float64(246.16690559453045), 'wc': np.float64(19.442229376915883), 'rc': np.float64(117.6242605207385

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e3ad2b0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(10.655838595793952), 'bp': np.float64(20.010236819882145), 'sg': np.float64(215.49980646808285), 'al': np.float64(896.8553459119493), 'su': np.float64(47.11007871983482), 'rbc': np.float64(63.37777777777779), 'pc': np.float64(165.98941798941797), 'pcc': np.float64(43.3015873015873), 'ba': np.float64(43.3015873015873), 'bgr': np.float64(56.268438004244295), 'bu': np.float64(106.02689573659777), 'sc': np.float64(126.1858727117594), 'sod': np.float64(81.79112984436362), 'pot': np.float64(4.282811103195126), 'hemo': np.float64(265.19343917266224), 'pcv': np.float64(246.16690559453045), 'wc': np.float64(19.442229376915883), 'rc': np.float64(117.6242605207385

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e347fb0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(10.655838595793952), 'bp': np.float64(20.010236819882145), 'sg': np.float64(215.49980646808285), 'al': np.float64(896.8553459119493), 'su': np.float64(47.11007871983482), 'rbc': np.float64(63.37777777777779), 'pc': np.float64(165.98941798941797), 'pcc': np.float64(43.3015873015873), 'ba': np.float64(43.3015873015873), 'bgr': np.float64(56.268438004244295), 'bu': np.float64(106.02689573659777), 'sc': np.float64(126.1858727117594), 'sod': np.float64(81.79112984436362), 'pot': np.float64(4.282811103195126), 'hemo': np.float64(265.19343917266224), 'pcv': np.float64(246.16690559453045), 'wc': np.float64(19.442229376915883), 'rc': np.float64(117.6242605207385

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e22c260>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(7.44545604641465), 'bp': np.float64(13.700947235708192), 'sg': np.float64(197.56958877642475), 'al': np.float64(835.2369237978095), 'su': np.float64(48.70931537598205), 'rbc': np.float64(80.47971781305114), 'pc': np.float64(251.4991181657849), 'pcc': np.float64(37.724867724867735), 'ba': np.float64(32.59428571428571), 'bgr': np.float64(75.89714788108624), 'bu': np.float64(104.46037047809341), 'sc': np.float64(127.64961732920857), 'sod': np.float64(79.43404525261117), 'pot': np.float64(0.6472020815416297), 'hemo': np.float64(303.2352212539296), 'pcv': np.float64(321.55957557453945), 'wc': np.float64(23.76221354152163), 'rc': np.float64(132.58131538025484

...

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f666f6162a0>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(14.057814049605197), 'bp': np.float64(17.28364583104029), 'sg': np.float64(194.49999587650058), 'al': np.float64(1190.9448818897642), 'su': np.float64(50.30621172353455), 'rbc': np.float64(60.36745406824148), 'pc': np.float64(261.5923009623796), 'pcc': np.float64(41.50262467191601), 'ba': np.float64(41.50262467191601), 'bgr': np.float64(77.14399863493536), 'bu': np.float64(113.80827083608544), 'sc': np.float64(145.9742818658552), 'sod': np.float64(78.80655655016432), 'pot': np.float64(4.058523803749385), 'hemo': np.float64(276.0173458789997), 'pcv': np.float64(265.12840140623865), 'wc': np.float64(17.771842750934134), 'rc': np.float64(133.88364686021492

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665e22e780>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(14.057814049605197), 'bp': np.float64(17.28364583104029), 'sg': np.float64(194.49999587650058), 'al': np.float64(1190.9448818897642), 'su': np.float64(50.30621172353455), 'rbc': np.float64(60.36745406824148), 'pc': np.float64(261.5923009623796), 'pcc': np.float64(41.50262467191601), 'ba': np.float64(41.50262467191601), 'bgr': np.float64(77.14399863493536), 'bu': np.float64(113.80827083608544), 'sc': np.float64(145.9742818658552), 'sod': np.float64(78.80655655016432), 'pot': np.float64(4.058523803749385), 'hemo': np.float64(276.0173458789997), 'pcv': np.float64(265.12840140623865), 'wc': np.float64(17.771842750934134), 'rc': np.float64(133.88364686021492

Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x7f665c90ca40>
DODA Pipeline Started
Running: SelectKBest
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of scores: 24
Operator Scores:
{'SelectKBest_1': {'age': np.float64(14.057814049605197), 'bp': np.float64(17.28364583104029), 'sg': np.float64(194.49999587650058), 'al': np.float64(1190.9448818897642), 'su': np.float64(50.30621172353455), 'rbc': np.float64(60.36745406824148), 'pc': np.float64(261.5923009623796), 'pcc': np.float64(41.50262467191601), 'ba': np.float64(41.50262467191601), 'bgr': np.float64(77.14399863493536), 'bu': np.float64(113.80827083608544), 'sc': np.float64(145.9742818658552), 'sod': np.float64(78.80655655016432), 'pot': np.float64(4.058523803749385), 'hemo': np.float64(276.0173458789997), 'pcv': np.float64(265.12840140623865), 'wc': np.float64(17.771842750934134), 'rc': np.float64(133.88364686021492

SENSITIVITY: completed 25/25 CV runs
SENSITIVITY CV PERFORMANCE SUMMARY


,Top_K,Method,Model,Acc_Mean,Acc_STD,F1_Mean,F1_STD,AUC_Mean,AUC_STD
0,5,ANOVA,LR,0.9898,0.0151,0.9799,0.0300,1.0,0.0
1,5,ANOVA,RF,0.9874,0.0158,0.9755,0.0307,1.0,0.0
2,5,ANOVA,XGB,0.9898,0.0151,0.9799,0.0300,1.0,0.0
3,5,DODA,LR,0.9924,0.0138,0.9849,0.0274,1.0,0.0
4,5,DODA,RF,0.9836,0.0161,0.9682,0.0313,1.0,0.0
5,5,DODA,XGB,0.9898,0.0151,0.9799,0.0300,1.0,0.0
6,10,ANOVA,LR,0.9936,0.0130,0.9873,0.0260,1.0,0.0
7,10,ANOVA,RF,0.9949,0.0120,0.9896,0.0243,1.0,0.0
8,10,ANOVA,XGB,0.9898,0.0151,0.9799,0.0300,1.0,0.0
9,10,DODA,LR,0.9949,0.0120,0.9896,0.0243,1.0,0.0


In [20]:
sensitivity_perf_test = wilcoxon_holm_test(
    sensitivity_cv_results, ["Top_K", "Model"], value_col="ROC_AUC"
)
print("=" * 70)
print("SENSITIVITY — ANOVA vs DODA ROC-AUC: SIGNIFICANCE TEST")
print("=" * 70)
display(sensitivity_perf_test.round(4))

SENSITIVITY — ANOVA vs DODA ROC-AUC: SIGNIFICANCE TEST


,Top_K,Model,ANOVA_mean,DODA_mean,p_value,cohens_d,p_holm,significant
0,5,LR,1.0,1.0,1.0,0.00,1.0,False
1,5,RF,1.0,1.0,1.0,0.00,1.0,False
2,5,XGB,1.0,1.0,1.0,-0.28,1.0,False
3,10,LR,1.0,1.0,1.0,0.00,1.0,False
4,10,RF,1.0,1.0,1.0,0.00,1.0,False
5,10,XGB,1.0,1.0,1.0,0.28,1.0,False
6,15,LR,1.0,1.0,1.0,0.00,1.0,False
7,15,RF,1.0,1.0,1.0,-0.28,1.0,False
8,15,XGB,1.0,1.0,1.0,0.00,1.0,False
9,20,LR,1.0,1.0,1.0,0.00,1.0,False


# Primary vs. Sensitivity — Head-to-Head Comparison

In [21]:
# =============================================================================
# SIDE-BY-SIDE: STABILITY SUMMARY
# =============================================================================

primary_stability_summary["Analysis"] = "Primary (imputed, n=400)"
sensitivity_stability_summary["Analysis"] = "Sensitivity (complete-case, n=158)"

stability_side_by_side = pd.concat(
    [primary_stability_summary, sensitivity_stability_summary], ignore_index=True
).pivot_table(index=["Top_K", "Method"], columns="Analysis", values="mean").round(4)

print("=" * 70)
print("MEAN JACCARD STABILITY — PRIMARY vs SENSITIVITY")
print("=" * 70)
display(stability_side_by_side)

MEAN JACCARD STABILITY — PRIMARY vs SENSITIVITY


Analysis      Primary (imputed, n=400)  Sensitivity (complete-case, n=158)
Top_K Method                                                              
5     ANOVA                     0.8489                              0.7833
      DODA                      0.8600                              0.7637
10    ANOVA                     0.8084                              0.9721
      DODA                      0.7838                              0.9108
15    ANOVA                     0.8968                              0.9280
      DODA                      0.8745                              0.9221
20    ANOVA                     0.9600                              0.9638
      DODA                      0.9613                              0.9848

In [22]:
# =============================================================================
# SIDE-BY-SIDE: PREDICTIVE PERFORMANCE (ROC-AUC) SUMMARY
# =============================================================================

primary_auc = primary_cv_summary[["Top_K", "Method", "Model", "AUC_Mean"]].copy()
primary_auc["Analysis"] = "Primary"
sensitivity_auc = sensitivity_cv_summary[["Top_K", "Method", "Model", "AUC_Mean"]].copy()
sensitivity_auc["Analysis"] = "Sensitivity"

auc_side_by_side = pd.concat([primary_auc, sensitivity_auc], ignore_index=True).pivot_table(
    index=["Top_K", "Method", "Model"], columns="Analysis", values="AUC_Mean"
).round(4)

print("=" * 70)
print("MEAN ROC-AUC — PRIMARY vs SENSITIVITY")
print("=" * 70)
display(auc_side_by_side)

auc_side_by_side.to_csv("../../../results/ckd/primary_vs_sensitivity_auc_comparison.csv")
stability_side_by_side.to_csv("../../../results/ckd/primary_vs_sensitivity_stability_comparison.csv")

MEAN ROC-AUC — PRIMARY vs SENSITIVITY


Analysis            Primary  Sensitivity
Top_K Method Model                      
5     ANOVA  LR      0.9955          1.0
             RF      0.9990          1.0
             XGB     0.9986          1.0
      DODA   LR      0.9948          1.0
             RF      0.9956          1.0
             XGB     0.9947          1.0
10    ANOVA  LR      0.9999          1.0
             RF      0.9996          1.0
             XGB     0.9983          1.0
      DODA   LR      0.9999          1.0
             RF      0.9994          1.0
             XGB     0.9980          1.0
15    ANOVA  LR      0.9999          1.0
             RF      0.9997          1.0
             XGB     0.9980          1.0
      DODA   LR      0.9999          1.0
             RF      0.9999          1.0
             XGB     0.9989          1.0
20    ANOVA  LR      1.0000          1.0
             RF      0.9999          1.0
             XGB     0.9990          1.0
      DODA   LR      1.0000          1.0
             RF      0.9999          1.0
             XGB     0.9988          1.0